## Building a simple SelfSupervisedLearner

In [ ]:
# dataset.py

from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision import transforms


class BirdDataset(Dataset):
    def __init__(self, image_dir, image_size=256):
        self.files = sorted(Path(image_dir).glob("*"))

        self.transform = transforms.Compose([
            transforms.Grayscale(),
            # transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx])

        img = self.transform(img)

        return img

In [4]:
# models.py

import torch
import torch.nn as nn


class AutoEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(128,64,2,stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,2,stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(32,1,2,stride=2),
            nn.Sigmoid()
        )

    def forward(self,x):

        z = self.encoder(x)

        out = self.decoder(z)

        return out

In [1]:
# train_autoencoder.py

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from dataset import BirdDataset
from models import AutoEncoder


device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = BirdDataset(r"C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\SelfSupervision\data\train")

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

model = AutoEncoder().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.MSELoss()

for epoch in range(30):

    running = 0

    for images in loader:

        images = images.to(device)

        outputs = model(images)

        loss = loss_fn(outputs, images)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running += loss.item()

    print(epoch, running/len(loader))

torch.save(model.state_dict(),"autoencoder.pt")

ModuleNotFoundError: No module named 'dataset'

In [2]:
import os
print(os.getcwd())

c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\SelfSupervision
